# Neural network pruning using Nonlinear Oblique Subspace ProjectionsExperiments for **MNIST** supporting:> J. Pena-Campos, A. Caicedo, C. Ocampo-Martinez, D. Patino,> *"Neural Network Pruning using Nonlinear Oblique Subspace Projections"*,> ACDSA 2026, Boracay, Philippines.> DOI: [10.1109/ACDSA67686.2026.11468014](https://doi.org/10.1109/ACDSA67686.2026.11468014)The idea under test is that **functional contribution**, measured by NObSP, identifies adifferent and better set of neurons to remove than **weight magnitude** does. The notebookprunes the same network by both criteria and compares accuracy as a function of how manyneurons are removed.---### AttributionThe functions `ObSP`, `NObSP_NN_single_MultiOutput_L1` and `NObSP_NN_single_MultiOutput_L2`implement the NObSP decomposition of> A. Caicedo et al., *Nonlinear Oblique Subspace Projections*,and are **(c) Alexander Caicedo, 2023**, reproduced here with permission. See `NOTICE` atthe repository root for the licence that applies to them. Everything else in this notebookis part of the thesis work.### Before runningSet `DATA_ROOT` in the first cell to the directory holding the trained checkpoints. Thenotebooks were originally run on Google Colab; the hard-coded Drive paths have beenreplaced by that constant.The datasets are **MNIST**, downloaded by `torchvision` on first run. No restricteddata is involved, and the checkpoints distributed alongside these notebooks were trainedon them.---### Authorship**Johan S. Peña-Campos** — PhD thesis, *"Interpretable Decision Support Systems for EarlyCancer Diagnosis and Personalized Treatment"*, Universitat Politècnica de Catalunya andPontificia Universidad Javeriana, 2026.### Use of AI assistanceGenerative AI (Claude, Anthropic) was used as a tool for code review, for verifying thecode against the results reported in the thesis, for detecting and correcting errors, andfor preparing this notebook for publication: stripping the embedded outputs, replacing thehard-coded Colab paths and writing the section headings. The research design, theexperiments and the interpretation of the results are the author's, who reviewed everychange and takes full responsibility for the content.

## 1. SetupImports and device selection.

In [ ]:
# Directorio con los checkpoints. Ajustar antes de ejecutar.
DATA_ROOT = "./"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import matplotlib.pyplot as plt

import numpy as np

from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader as tf_dataloader
import pandas as pd
import seaborn as sns
from torch.utils.data import random_split

import copy

## 2. The NObSP decompositionOblique projection onto the subspace spanned by each neuron's response, along the reference subspace spanned by the others. This is what separates a neuron's *functional* contribution from the part it shares with the rest of the layer, and it is why orthogonal projection would not do: the learned subspaces overlap.

In [ ]:
def ObSP(X,Y):
  # Function to compute the oblique projection onto X along the reference space defined by Y.
  # The input data is the following:
  # X: a matriz of size Nxd, containing the basis for the subspace where the data will be projected
  # Y: a matrix of size Nxd, containing the basis for the reference subspace
  #
  # The function returns the oblique projection matrix of size NxN.

  # Converting the input data into torch tensors

  if not torch.is_tensor(X):
    X = torch.from_numpy(X)

  if not torch.is_tensor(Y):
    Y = torch.from_numpy(Y)

  N, d = X.size() # computing the size of X

  P = Y@torch.linalg.pinv(torch.t(Y)@Y)@torch.t(Y) # Conputing the orthogonal projection matriz onto the subsapce given by Y
  Q = torch.eye(N,N)-P # Computing the complement of P

  P_xy = X@torch.linalg.pinv(torch.t(X)@Q@X)@torch.t(X)@torch.t(Q) # Computing the oblique projection matriz onto X along Y

  return P_xy

def NObSP_NN_single_MultiOutput_L1(X, L_out):
  N = np.size(X,0) # computing the size of X along dimension 0
  d = np.size(X,1) # computing the size of X along dimension 1

  #print("\nDimensión 0 en X:", N)
  #print("\nDimensión 1 en X:", d)

  p = list(model.children())[-1].out_features # computing the size of X along dimension 1

  #print("\nDimensión de p:", p)

  P_xy = np.zeros((N,N,d)) # Initializing proyection matrices
  y_e = torch.from_numpy(np.zeros((N,d,p))).type(torch.float) # Initializing Matriz where the estimated nonlinear contributions will be stored, converting to a tensor object
  neurons_last = list(model.children())[-1].in_features # Obtaining the dimension of the subspace where the data lies (number of neurons in the last layer)
  Alpha = torch.zeros(neurons_last,d*p) # Initializing the matrix for the Alpha coefficients, out-of-sample extension, using the dimension where the transformed data lies

  # Computing the transformation of the input data using inference mode, and extracting the evaluation of the input on the network up to the last layer.
  #with torch.inference_mode():
  # outputs_f, L3_f, L2_f, L1_f, L_In_f = model(X)

  P_x_target = torch.linalg.pinv(torch.t(X)@X)@torch.t(X) # Finding the projection matrix onto the matrix tused to find the alpha coeficients, out-of-sample extension
  print("\nDimensión de P_x_target:", P_x_target.shape)

  for l in range(p):
    for i in range(d):
      # Defining the input matrix that will be used to find the subspace of the nonlinear transformation of
      # the input variables x_i, onto which the output will be projected
      X_target = np.zeros((N,d))
      X_target[:, i] = X[:, i].cpu().numpy()

      # Defining the input matrix that will be used to find the reference subspace, along which the data
      # will be projected.
      X_reference = X.cpu().numpy().copy()
      X_reference[:,i] = 0

      # transforming the matrices to tensor objects to be used in pytorch
      X_target = torch.from_numpy(X_target).type(torch.float)
      X_reference = torch.from_numpy(X_reference).type(torch.float)

      # Computing the transformation of the input data using inference mode in order to find the basis for the susbpace of the transformations
      with torch.inference_mode():
        y_target, X_target_sub3, X_target_sub2  = model.forward_2(X_target)
        y_reference, X_reference_sub3, X_reference_sub2 = model.forward_2(X_reference)

      # Centering the bassis of the target and reference subspaces
      X_target_sub = (X_target-torch.mean(X_target,dim=0))
      X_reference_sub = (X_reference-torch.mean(X_reference,dim=0))

      # Computing the oblique projection onto the susbspace defined by the nonlienar transformation of x_i along
      # the reference subspace, which contains the nonlinear transofrmation of all variables except x_i
      P_xy[:,:,i] = ObSP(X_target_sub,X_reference_sub)
      P = torch.from_numpy(P_xy[:,:,i]).type(torch.float) # Converting form pytorch to numpy
      y_e[:,[i],[l]] = torch.unsqueeze(P@(L_out[:,l]-torch.mean(L_out[:,l])),-1) # Using the projection matrices to ptoject the output vector and find the nonlinear contribution of each variable.

    #print(f'initial column {l*d} final column {l*d+d}')

  #print(f'Size Alpha {y_e.size()}')
  return P_xy, y_e

def NObSP_NN_single_MultiOutput_L2(X, L_out):
  N = np.size(X,0) # computing the size of X along dimension 0
  d = np.size(X,1) # computing the size of X along dimension 1

  #print("\nDimensión 0 en X:", N)
  #print("\nDimensión 1 en X:", d)

  p = list(model.children())[-1].out_features # computing the size of X along dimension 1

  #print("\nDimensión de p:", p)

  P_xy = np.zeros((N,N,d)) # Initializing proyection matrices
  y_e = torch.from_numpy(np.zeros((N,d,p))).type(torch.float) # Initializing Matriz where the estimated nonlinear contributions will be stored, converting to a tensor object
  neurons_last = list(model.children())[-1].in_features # Obtaining the dimension of the subspace where the data lies (number of neurons in the last layer)
  Alpha = torch.zeros(neurons_last,d*p) # Initializing the matrix for the Alpha coefficients, out-of-sample extension, using the dimension where the transformed data lies

  # Computing the transformation of the input data using inference mode, and extracting the evaluation of the input on the network up to the last layer.
  #with torch.inference_mode():
  # outputs_f, L3_f, L2_f, L1_f, L_In_f = model(X)

  P_x_target = torch.linalg.pinv(torch.t(X)@X)@torch.t(X) # Finding the projection matrix onto the matrix tused to find the alpha coeficients, out-of-sample extension
  print("\nDimensión de P_x_target:", P_x_target.shape)

  for l in range(p):
    for i in range(d):
      # Defining the input matrix that will be used to find the subspace of the nonlinear transformation of
      # the input variables x_i, onto which the output will be projected
      X_target = np.zeros((N,d))
      X_target[:, i] = X[:, i].cpu().numpy()

      # Defining the input matrix that will be used to find the reference subspace, along which the data
      # will be projected.
      X_reference = X.cpu().numpy().copy()
      X_reference[:,i] = 0

      # transforming the matrices to tensor objects to be used in pytorch
      X_target = torch.from_numpy(X_target).type(torch.float)
      X_reference = torch.from_numpy(X_reference).type(torch.float)

      # Computing the transformation of the input data using inference mode in order to find the basis for the susbpace of the transformations
      with torch.inference_mode():
        y_target, X_target_sub3  = model.forward_3(X_target)
        y_reference, X_reference_sub3 = model.forward_3(X_reference)

      # Centering the bassis of the target and reference subspaces
      X_target_sub = (X_target-torch.mean(X_target,dim=0))
      X_reference_sub = (X_reference-torch.mean(X_reference,dim=0))

      # Computing the oblique projection onto the susbspace defined by the nonlienar transformation of x_i along
      # the reference subspace, which contains the nonlinear transofrmation of all variables except x_i
      P_xy[:,:,i] = ObSP(X_target_sub,X_reference_sub)
      P = torch.from_numpy(P_xy[:,:,i]).type(torch.float) # Converting form pytorch to numpy
      y_e[:,[i],[l]] = torch.unsqueeze(P@(L_out[:,l]-torch.mean(L_out[:,l])),-1) # Using the projection matrices to ptoject the output vector and find the nonlinear contribution of each variable.

    #print(f'initial column {l*d} final column {l*d+d}')

  #print(f'Size Alpha {y_e.size()}')
  return P_xy, y_e

## 3. Network and dataA multilayer perceptron on MNIST. `fc1` has 512 units and is the layer whose neurons are ranked and pruned.

In [ ]:
# Definir la arquitectura de la red MLP profunda
class NetImprovedMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 300)
        self.bn1 = nn.BatchNorm1d(300)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(300, 150)
        self.bn2 = nn.BatchNorm1d(150)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(150, 50)
        self.bn3 = nn.BatchNorm1d(50)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(50, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x1 = self.dropout1(self.relu1(self.bn1(self.fc1(x))))
        x2 = self.dropout2(self.relu2(self.bn2(self.fc2(x1))))
        x3 = self.relu3(self.bn3(self.fc3(x2)))
        x4 = self.fc4(x3)
        prob = F.softmax(x4, dim=1)
        return prob, x4, x3, x2, x1, x

    def forward_2(self, x1):
        x2 = self.dropout2(self.relu2(self.bn2(self.fc2(x1))))
        x3 = self.relu3(self.bn3(self.fc3(x2)))
        x4 = self.fc4(x3)
        return x4, x3, x2

    def forward_3(self, x2):
        x3 = self.relu3(self.bn3(self.fc3(x2)))
        x4 = self.fc4(x3)
        return x4, x3

In [ ]:
# Definir hiperparámetros
batch_size = 128
learning_rate = 0.001
num_epochs = 100
patience = 10  # Para early stopping

# Cargar el dataset MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Dividir el dataloader en entrenamiento, validación y prueba
train_size = int(0.7 * len(full_trainset))
val_size = len(full_trainset) - train_size
train_subset, val_subset = random_split(full_trainset, [train_size, val_size])

trainloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

# Definir el tamaño de la entrada y el número de clases
input_size = 28 * 28
num_classes = 10

In [ ]:
# Instanciar el modelo
model = NetImprovedMLP(input_size, num_classes)

# Definir la función de pérdida y el optimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Early stopping variables
best_val_loss = float('inf')
epochs_no_improve = 0
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Entrenamiento del modelo con early stopping
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for batch_idx, (data, target) in enumerate(trainloader):
        optimizer.zero_grad()
        outputs, _, _, _, _, _ = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += target.size(0)
        correct_train += (predicted == target).sum().item()

    epoch_loss_train = running_loss / len(trainloader)
    epoch_accuracy_train = 100 * correct_train / total_train
    train_losses.append(epoch_loss_train)
    train_accuracies.append(epoch_accuracy_train)

    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for data, target in valloader:
            outputs, _, _, _, _, _ = model(data)
            loss = criterion(outputs, target)
            running_val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_val += target.size(0)
            correct_val += (predicted == target).sum().item()

    epoch_loss_val = running_val_loss / len(valloader)
    epoch_accuracy_val = 100 * correct_val / total_val
    val_losses.append(epoch_loss_val)
    val_accuracies.append(epoch_accuracy_val)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss_train:.4f}, Train Acc: {epoch_accuracy_train:.2f}%, Val Loss: {epoch_loss_val:.4f}, Val Acc: {epoch_accuracy_val:.2f}%')

    if epoch_loss_val < best_val_loss:
        best_val_loss = epoch_loss_val
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping triggered at epoch {epoch+1}')
            model.load_state_dict(best_model_state)
            break

In [ ]:
ruta_guardado_completo = DATA_ROOT + 'modelo_completo_Ori_MNIST.pth'

In [ ]:
# Evaluación final en el conjunto de prueba
model.eval()
correct_test = 0
total_test = 0
with torch.no_grad():
    for data, target in testloader:
        outputs, _, _, _, _, _ = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total_test += target.size(0)
        correct_test += (predicted == target).sum().item()

accuracy_test = 100 * correct_test / total_test
print(f'\nTest Accuracy: {accuracy_test:.2f}%')

# Graficar la pérdida y la precisión durante el entrenamiento y la validación
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss during Training')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy during Training')
plt.legend()
plt.show()

# Guardar el modelo completo
torch.save(model, ruta_guardado_completo)

print(f"Modelo completo guardado exitosamente en: {ruta_guardado_completo}")

In [ ]:
# Cargar el modelo completo
try:
    model = torch.load(DATA_ROOT + "modelo_completo_Ori_MNIST.pth", weights_only=False)
    print("Modelo cargado exitosamente.")

    # Si vas a usar el modelo para inferencia, es recomendable ponerlo en modo de evaluación
    model.eval()
    print("Modelo puesto en modo de evaluación.")

except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta: {ruta_guardado_completo}")
except Exception as e:
    print(f"Ocurrió un error al cargar el modelo: {e}")

In [ ]:
print(f"Tamaño total del dataset de prueba (Train): {len(trainloader.dataset)}")
print(f"Tamaño total del dataset de prueba (Validation): {len(valloader.dataset)}")
print(f"Tamaño total del dataset de prueba (Test): {len(testloader.dataset)}")

## 4. Extracting the layer responsesThe decomposition operates on the responses of the target layer over a fixed subset of inputs, and on the weights of the layer that follows it.

In [ ]:
# Acceder a los pesos de la capa

first_linear_layer = model.fc1
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (300 neuronas):", weights_layer1.shape)
print("Forma de los biases de la primera capa lineal (300 neuronas):", biases_layer1.shape)

first_linear_layer = model.fc2
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (150 neuronas):", weights_layer1.shape)
print("Forma de los biases de la primera capa lineal (150 neuronas):", biases_layer1.shape)

first_linear_layer = model.fc3
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (50 neuronas):", weights_layer1.shape)
print("Forma de los biases de la primera capa lineal (50 neuronas):", biases_layer1.shape)

# Imprimir algunos valores de los pesos (opcional - comentar para evitar salida larga)
# print("\nAlgunos valores de los pesos de la primera capa lineal:\n", weights_layer1[:5, :5])

In [ ]:
from torch.utils.data import Subset

num_images = 600
indices = list(range(num_images))
subset = Subset(testset, indices)

# Inicializar listas para almacenar las imágenes y las etiquetas
images_list = []
labels_list = []

# Iterar a través del subset y agregar las imágenes y etiquetas a las listas
for img, label in subset:
    images_list.append(img)
    labels_list.append(label)

# Convertir las listas de imágenes y etiquetas a tensores de PyTorch
all_images = torch.stack(images_list)
all_labels = torch.tensor(labels_list)

print("Forma del tensor de todas las imágenes:", all_images.shape)
print("Forma del tensor de todas las etiquetas:", all_labels.shape)

In [ ]:
model = torch.load(DATA_ROOT + "modelo_completo_Ori_MNIST.pth", weights_only=False)
# Evaluar el modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
  L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
  _, predicted = torch.max(L_out.data, 1)
  total += all_labels.size(0)
  correct += (predicted == all_labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

In [ ]:
print("\nForma de los pesos de la capa 1 lineal (512 neuronas):", L_In.shape)
print("\nForma de los pesos de la capa 2 lineal (512 neuronas):", L1.shape)
print("\nForma de los pesos de la capa 3 lineal (512 neuronas):", L2.shape)
print("\nForma de los pesos de la capa 4 lineal (512 neuronas):", L3.shape)
print("\nForma de los pesos de la capa 4 lineal (512 neuronas):", L4.shape)
print("\nForma de los pesos de la capa 4 lineal (512 neuronas):", L_out.shape)

## 5. Contribution of each neuron`NObSP_NN_single_MultiOutput_L1` returns, for every neuron, the component of the output attributable to it alone. Neurons are then ranked by the magnitude of that component.

In [ ]:
Pxy_L1, y_e_L1 = NObSP_NN_single_MultiOutput_L1(L1, L_out)
Pxy_L2, y_e_L2 = NObSP_NN_single_MultiOutput_L2(L2, L_out)

In [ ]:
import pickle

root_N = DATA_ROOT + ''
nombre_archivo = "Proyecciones_YEst_MNIST.pkl"

with open(root_N+nombre_archivo, 'wb') as archivo:
  pickle.dump([Pxy_L1, y_e_L1, Pxy_L2, y_e_L2], archivo)

In [ ]:
Cont_NL1_sorted = torch.from_numpy(np.zeros((300,2,10))).type(torch.float)
Cont_NL2_sorted = torch.from_numpy(np.zeros((150,2,10))).type(torch.float)

indices_L1 = torch.arange(1, Cont_NL1_sorted.shape[0]+1)  # Crear un tensor de índices del 1 al 300
indices_L2 = torch.arange(1, Cont_NL2_sorted.shape[0]+1)  # Crear un tensor de índices del 1 al 300

for i in range(Cont_NL1_sorted.shape[2]):
  Cont_N=torch.diag(torch.t(y_e_L1[:,:,i])@y_e_L1[:,:,i])/torch.max(torch.diag(torch.t(y_e_L1[:,:,i])@y_e_L1[:,:,i]))
  Cont_Ni = torch.stack((indices_L1.float(), Cont_N), dim=1) # Ahora Cont_N_with_indices tiene forma [300, 2], donde la primera columna son los índices y la segunda son los valores de Cont_N

  # Organizar de mayor a menor según Cont_N sin perder el índice original
  # Obtener los índices ordenados por el valor de Cont_N (segunda columna)
  sorted_indices = torch.argsort(Cont_N, descending=True)

  # Reorganizar Cont_N_with_indices usando los índices ordenados
  Cont_NL1_sorted[:,:,i] = Cont_Ni[sorted_indices]

  # Graficar la variable ordenada y los índices
  # Extraer los índices y los valores ordenados para graficar
  sorted_indices_for_plot = Cont_NL1_sorted[:, 0, i]
  sorted_values_for_plot = Cont_NL1_sorted[:, 1, i]

  # Crear la gráfica
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(indices_L1.numpy(), sorted_values_for_plot.numpy(), marker='o', linestyle='-')
  plt.xlabel('Índice Original')
  plt.ylabel('Valor de Contribución Neurona')
  plt.title('Valores de Contribución N=300 Clase: '+str(i))
  plt.xticks(sorted_indices_for_plot.numpy())  # Establecer los ticks del eje X como los índices
  plt.grid(True)
  plt.show()

  Cont_N=torch.diag(torch.t(y_e_L2[:,:,i])@y_e_L2[:,:,i])/torch.max(torch.diag(torch.t(y_e_L2[:,:,i])@y_e_L2[:,:,i]))
  Cont_Ni = torch.stack((indices_L2.float(), Cont_N), dim=1) # Ahora Cont_N_with_indices tiene forma [300, 2], donde la primera columna son los índices y la segunda son los valores de Cont_N

  # Organizar de mayor a menor según Cont_N sin perder el índice original
  # Obtener los índices ordenados por el valor de Cont_N (segunda columna)
  sorted_indices = torch.argsort(Cont_N, descending=True)

  # Reorganizar Cont_N_with_indices usando los índices ordenados
  Cont_NL2_sorted[:,:,i] = Cont_Ni[sorted_indices]

  # Graficar la variable ordenada y los índices
  # Extraer los índices y los valores ordenados para graficar
  sorted_indices_for_plot = Cont_NL2_sorted[:, 0, i]
  sorted_values_for_plot = Cont_NL2_sorted[:, 1, i]

  # Crear la gráfica
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(indices_L2.numpy(), sorted_values_for_plot.numpy(), marker='o', linestyle='-')
  plt.xlabel('Índice Original')
  plt.ylabel('Valor de Contribución Neurona')
  plt.title('Valores de Contribución N=150 Clase: '+str(i))
  plt.xticks(sorted_indices_for_plot.numpy())  # Establecer los ticks del eje X como los índices
  plt.grid(True)
  plt.show()

In [ ]:
# Crear la matriz de concurrencia
concurrencia_matrix_L1 = np.zeros((300), dtype=float)
neu_i=0
# Llenar la matriz de concurrencia
for neu_i in range(300):
    for pos_i in range(10):
        # Contar cuántas veces la neurona neu_i aparece en la posición pos_i para todas las clases
        concurrencia_matrix_L1[neu_i]=concurrencia_matrix_L1[neu_i].item()+Cont_NL1_sorted[Cont_NL1_sorted[:,0,pos_i] == neu_i+1,1,pos_i].item()

print("Forma de la matriz de concurrencia:", concurrencia_matrix_L1.shape)
#print(concurrencia_matrix_L1)

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.stem(indices_L1.numpy(), concurrencia_matrix_L1, linefmt='-', markerfmt='o', basefmt=' ')
plt.xlabel('Índice Original')
plt.ylabel('Valor de Contribución Neurona')
plt.title('Valores de Contribución N=300')
plt.grid(True)
plt.show()

# Crear la matriz de concurrencia
concurrencia_matrix_L2 = np.zeros((150), dtype=float)

# Llenar la matriz de concurrencia
for neu_i in range(150):
    for pos_i in range(10):
        # Contar cuántas veces la neurona neu_i aparece en la posición pos_i para todas las clases
        concurrencia_matrix_L2[neu_i]=concurrencia_matrix_L2[neu_i].item()+Cont_NL1_sorted[Cont_NL1_sorted[:,0,pos_i] == neu_i+1,1,pos_i].item()

print("Forma de la matriz de concurrencia:", concurrencia_matrix_L2.shape)
#print(concurrencia_matrix_L1)

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.stem(indices_L2.numpy(), concurrencia_matrix_L2, linefmt='-', markerfmt='o', basefmt=' ')
plt.xlabel('Índice Original')
plt.ylabel('Valor de Contribución Neurona')
plt.title('Valores de Contribución N=150')
plt.grid(True)
plt.show()

In [ ]:
print("El valor minimo es: ",np.min(concurrencia_matrix_L1))
print("Esta ubicado en la posición: ",np.argmin(concurrencia_matrix_L1))
print("Verificando el resultado: ",concurrencia_matrix_L1[np.argmin(concurrencia_matrix_L1)])

print("El valor minimo es: ",np.min(concurrencia_matrix_L2))
print("Esta ubicado en la posición: ",np.argmin(concurrencia_matrix_L2))
print("Verificando el resultado: ",concurrencia_matrix_L2[np.argmin(concurrencia_matrix_L2)])

# Reorganizar Cont_N_with_indices usando los índices ordenados
indices_ordenamiento = np.argsort(concurrencia_matrix_L2)[::-1]
indices_ordenamiento[-1]
np.min(indices_ordenamiento)
np.max(indices_ordenamiento)
contribuciones_ordenadas = concurrencia_matrix_L2[indices_ordenamiento]

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(contribuciones_ordenadas, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Valores de Contribución N=150')
plt.xticks(indices_ordenamiento)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

suma_acumulada = np.cumsum(contribuciones_ordenadas)
print("Suma acumulada:", suma_acumulada[-1])
print("Suma total:", np.sum(contribuciones_ordenadas))

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(suma_acumulada, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Suma acumulada N=150')
plt.xticks(indices_ordenamiento)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

Por_Dej=0.9

print("Suma total:", Por_Dej*np.sum(contribuciones_ordenadas))
suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas)
indices_ordenamiento[np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]]

print("La longitud de posiciones inferiores al 90% es: ",len(np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]))

indices_ordenamiento_L1 = np.argsort(concurrencia_matrix_L1)[::-1]
indices_ordenamiento_L1[-1]
np.min(indices_ordenamiento_L1)
np.max(indices_ordenamiento_L1)
contribuciones_ordenadas_L1 = concurrencia_matrix_L1[indices_ordenamiento_L1]

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(contribuciones_ordenadas_L1, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Valores de Contribución N=300')
plt.xticks(indices_ordenamiento_L1)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

suma_acumulada_L1 = np.cumsum(contribuciones_ordenadas_L1)
print("Suma acumulada:", suma_acumulada_L1[-1])
print("Suma total:", np.sum(contribuciones_ordenadas_L1))

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(suma_acumulada_L1, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Suma acumulada N=300')
plt.xticks(indices_ordenamiento_L1)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

print("Suma total:", Por_Dej*np.sum(contribuciones_ordenadas_L1))
suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1)
indices_ordenamiento_L1[np.where(suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1))[0]]

print("La longitud de posiciones inferiores al 90% es: ",len(np.where(suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1))[0]))

## 6. Removing a neuron`Quitar_Contri` zeroes a neuron and returns the modified network, so that accuracy can be measured as neurons are removed one at a time.

In [ ]:
def Quitar_Contri(model_Modif, indice, model_Ori, capa):
  print("Se quita el indice: ",indice)

  if capa == 2:
    first_linear_layer = model_Modif.fc2
    N = '150'
  elif capa ==1:
    first_linear_layer = model_Modif.fc1
    N = '300'

  weights_layer1 = first_linear_layer.weight.data.numpy()
  biases_layer1 = first_linear_layer.bias.data.numpy()

  print("\nForma de los pesos de la primera capa lineal ("+N+" neuronas):", weights_layer1.shape)
  plt.figure(figsize=(30, 6))
  plt.imshow(weights_layer1, cmap='gray')
  plt.axis('off')  # Para no mostrar los ejes
  plt.title('Pesos')
  plt.show()

  print("Forma de los biases de la primera capa lineal ("+N+" neuronas):", biases_layer1.shape)
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(biases_layer1, marker='o', linestyle='-')
  plt.grid(True)
  plt.show()

  weights_layer1[indice,:] = 0
  biases_layer1[indice] = 0

  if capa == 2:
    model_Modif.fc2.weight.data = torch.from_numpy(weights_layer1)
    model_Modif.fc2.bias.data = torch.from_numpy(biases_layer1)
  elif capa ==1:
    model_Modif.fc1.weight.data = torch.from_numpy(weights_layer1)
    model_Modif.fc1.bias.data = torch.from_numpy(biases_layer1)


  print("\nForma de los pesos de la primera capa lineal ("+N+" neuronas):", weights_layer1.shape)
  plt.figure(figsize=(30, 6))
  plt.imshow(weights_layer1, cmap='gray')
  plt.axis('off')  # Para no mostrar los ejes
  plt.title('Pesos')
  plt.show()
  print("Forma de los biases de la primera capa lineal ("+N+" neuronas):", biases_layer1.shape)
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(biases_layer1, marker='o', linestyle='-')
  plt.grid(True)
  plt.show()

  # Evaluar el modelo
  model_Ori.eval()
  correct_r = 0
  total = 0
  with torch.no_grad():
    L_out, L4, L3, L2, L1, L_In = model_Ori(all_images) # Desempaquetar las salidas
    _, predicted = torch.max(L_out.data, 1)
    total += all_labels.size(0)
    correct_r += (predicted == all_labels).sum().item()

  print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct_r / total:.2f} %')

  # Evaluar el modelo
  model_Modif.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    L_out, L4, L3, L2, L1, L_In = model_Modif(all_images) # Desempaquetar las salidas
    _, predicted = torch.max(L_out.data, 1)
    total += all_labels.size(0)
    correct += (predicted == all_labels).sum().item()

  print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

  return model_Modif, 100 * correct / total, 100 * correct_r / total

In [ ]:
def Quitar_Contri_NVerbo(model_Mod, indice, capa):
  if capa == 2:
    first_linear_layer = model_Mod.fc2
    N = '150'
  elif capa ==1:
    first_linear_layer = model_Mod.fc1
    N = '300'

  weights_layer1 = first_linear_layer.weight.data.numpy()
  biases_layer1 = first_linear_layer.bias.data.numpy()

  weights_layer1[indice,:] = 0
  biases_layer1[indice] = 0

  if capa == 2:
    model_Mod.fc2.weight.data = torch.from_numpy(weights_layer1)
    model_Mod.fc2.bias.data = torch.from_numpy(biases_layer1)
  elif capa ==1:
    model_Mod.fc1.weight.data = torch.from_numpy(weights_layer1)
    model_Mod.fc1.bias.data = torch.from_numpy(biases_layer1)

  # Evaluar el modelo
  model_Mod.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    L_out, L4, L3, L2, L1, L_In = model_Mod(all_images) # Desempaquetar las salidas
    _, predicted = torch.max(L_out.data, 1)
    total += all_labels.size(0)
    correct += (predicted == all_labels).sum().item()

  return model_Mod, 100 * correct / total

In [ ]:
model = torch.load(ruta_guardado_completo, weights_only=False)
model_mod4 = copy.deepcopy(model)

model_mod4, Perf_L1_2, Perf_L1_Org1  = Quitar_Contri(model_mod4, indices_ordenamiento_L1[np.where(suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1))[0]], model, 1)
model_mod4, Perf_L2_2, Perf_L2_Org1  = Quitar_Contri(model_mod4, indices_ordenamiento[np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]], model, 2)
len(indices_ordenamiento[np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]])

In [ ]:
model = torch.load(ruta_guardado_completo, weights_only=False)
model_mod5 = copy.deepcopy(model)

cont_i = 0
cont_j = 0
Perf_L1_L2 = np.zeros((len(range(0, -101, -1)), len(range(0, -151, -1))), dtype=float)
indices_ordenamiento_L1[-1]
for i_N in range(0, -101, -1):
  print("Voy en: ", i_N)
  if i_N !=0:
    model = torch.load(ruta_guardado_completo, weights_only=False)
    first_linear_layer = model.fc1
    weights_layer1 = first_linear_layer.weight.data.numpy()
    biases_layer1 = first_linear_layer.bias.data.numpy()
    model_mod5.fc1.weight.data = torch.from_numpy(weights_layer1)
    model_mod5.fc1.bias.data = torch.from_numpy(biases_layer1)
    model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N], 2)
  for j_N in range(0, -151, -1):

    if i_N == 0 and j_N == 0:
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[cont_i,cont_j]= 100 * correct / total
    else:
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N], 1)
    cont_j+=1
  cont_i+=1
  cont_j=0

from matplotlib import rcParams

rcParams.update({'font.size': 14})
from matplotlib.ticker import MultipleLocator

fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2.shape[1] - 1, Perf_L1_L2.shape[1])
y = np.linspace(0, Perf_L1_L2.shape[0] - 1, Perf_L1_L2.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance', labelpad=10)

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

ax.xaxis.set_major_locator(MultipleLocator(50))

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

Perf_L1_L2

nombre_archivo = DATA_ROOT + 'perf_prun_NObSP_NMIST.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()

## 7. The comparisonThe same network is pruned twice: once by the NObSP ranking, once by L1 and L2 weight magnitude using `torch.nn.utils.prune`. Accuracy is recorded against the number of neurons removed.

In [ ]:
import torch.nn.utils.prune as prune

cont_i = 0
cont_j = 0
Perf_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151))), dtype=float)

# for i_N in range(0, 30, 1):
for i_N in range(0, 101, 1):
  print("Voy en: ", i_N)
  if i_N !=0:
    model = torch.load(ruta_guardado_completo, weights_only=False)
    model_mod5 = copy.deepcopy(model)
    pruning_fraction_i = i_N/150
    # Aplicar pruning estructurado (por fila) a fc2
    prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=2, dim=0)
    with torch.no_grad():
        weight_mask = model_mod5.fc2.weight_mask
        if model_mod5.fc2.bias is not None:
            bias_mask = model_mod5.fc2.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc2.bias.data = bias_mask
    prune.remove(model_mod5.fc2, 'weight')
    model_mod5.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
      _, predicted = torch.max(L_out.data, 1)
      total += all_labels.size(0)
      correct += (predicted == all_labels).sum().item()
    Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
  else:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
      _, predicted = torch.max(L_out.data, 1)
      total += all_labels.size(0)
      correct += (predicted == all_labels).sum().item()
    Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
  cont_j+=1
  # for j_N in range(0, 50, 1):
  for j_N in range(1, 151, 1):
    # if i_N == 0 and j_N == 0:
    #   model.eval()
    #   correct = 0
    #   total = 0
    #   with torch.no_grad():
    #     L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
    #     _, predicted = torch.max(L_out.data, 1)
    #     total += all_labels.size(0)
    #     correct += (predicted == all_labels).sum().item()
    #   Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    # elif i_N == 0 and j_N != 0:
    if i_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=2, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = model_mod5.fc1.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    else:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      # model_mod5 = copy.deepcopy(model)
      # prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=2, dim=0)
      # with torch.no_grad():
      #   weight_mask = model_mod5.fc2.weight_mask
      #   if model_mod5.fc2.bias is not None:
      #     bias_mask = model_mod5.fc2.bias.data
      #     for i in range(weight_mask.size(0)):
      #       if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
      #         bias_mask[i] = 0
      #     model_mod5.fc2.bias.data = bias_mask
      # prune.remove(model_mod5.fc2, 'weight')
      first_linear_layer = model.fc1
      weights_layer1 = first_linear_layer.weight.data.numpy()
      biases_layer1 = first_linear_layer.bias.data.numpy()
      model_mod5.fc1.weight.data = torch.from_numpy(weights_layer1)
      model_mod5.fc1.bias.data = torch.from_numpy(biases_layer1)
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=2, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = model_mod5.fc1.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    cont_j+=1
  cont_i+=1
  cont_j=0

rcParams.update({'font.size': 14})

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2_P.shape[1] - 1, Perf_L1_L2_P.shape[1])
y = np.linspace(0, Perf_L1_L2_P.shape[0] - 1, Perf_L1_L2_P.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2_P, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance')

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

ax.xaxis.set_major_locator(MultipleLocator(50))

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

nombre_archivo = DATA_ROOT + 'perf_prun_pyTorch_L2_MNIST.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()

model.fc1.weight.data.numpy()
plt.figure(figsize=(30, 50))
plt.imshow(model_mod5.fc2.weight.data.numpy(), cmap='gray')
plt.axis('off')  # Para no mostrar los ejes
plt.title('Pesos')
plt.show()

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(model_mod5.fc2.bias.data.numpy(), marker='o', linestyle='-')
plt.grid(True)
plt.show()

In [ ]:

fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(X, Y, Perf_L1_L2_P-Perf_L1_L2, cmap='viridis')
ax.plot_surface(X, Y, np.zeros((len(range(0, 101)), len(range(0, 151))), dtype=float), cmap='viridis')


# 4. Añadir etiquetas y título
ax.set_xlabel('N 300')
ax.set_ylabel('N 150')
ax.set_zlabel('Performance')

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 30   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

# 6. Mostrar la gráfica
plt.show()

In [ ]:
model = torch.load(ruta_guardado_completo, weights_only=False)
model_mod5 = copy.deepcopy(model)

cont_i = 0
cont_j = 0
Perf_L1_L2 = np.zeros((len(range(0, -101, -1)), len(range(0, -151, -1))), dtype=float)
indices_ordenamiento_L1[-1]
for i_N in range(0, -101, -1):
  print("Voy en: ", i_N)
  for j_N in range(0, -151, -1):

    if i_N == 0 and j_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[cont_i,cont_j]= 100 * correct / total
    elif i_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
    elif j_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
    else:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      model_mod5, _ = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
    cont_j+=1
  cont_i+=1
  cont_j=0

rcParams.update({'font.size': 14})

fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2.shape[1] - 1, Perf_L1_L2.shape[1])
y = np.linspace(0, Perf_L1_L2.shape[0] - 1, Perf_L1_L2.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance', labelpad=10)

ax.xaxis.set_major_locator(MultipleLocator(50))

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

Perf_L1_L2[:,0]

nombre_archivo = DATA_ROOT + 'perf_prun_NObSP_NMIST.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()

In [ ]:
import torch.nn.utils.prune as prune

cont_i = 0
cont_j = 0
Perf_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151))), dtype=float)

# for i_N in range(0, 30, 1):
for i_N in range(0, 101, 1):
  print("Voy en: ", i_N)
  # for j_N in range(0, 50, 1):
  for j_N in range(0, 151, 1):
    if i_N == 0 and j_N == 0:
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    elif i_N == 0 and j_N != 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=2, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = model_mod5.fc1.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    else:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      pruning_fraction_i = i_N/150
      prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=2, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc2.weight_mask
        if model_mod5.fc2.bias is not None:
          bias_mask = model_mod5.fc2.bias.data
          for i in range(weight_mask.size(0)):
            if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
              bias_mask[i] = 0
          model_mod5.fc2.bias.data = bias_mask
      prune.remove(model_mod5.fc2, 'weight')
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=2, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = model_mod5.fc1.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    cont_j+=1
  cont_i+=1
  cont_j=0

rcParams.update({'font.size': 14})

fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2_P.shape[1] - 1, Perf_L1_L2_P.shape[1])
y = np.linspace(0, Perf_L1_L2_P.shape[0] - 1, Perf_L1_L2_P.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2_P, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance', labelpad=10)

ax.yaxis.set_major_locator(MultipleLocator(50))

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

nombre_archivo = DATA_ROOT + 'perf_prun_pyTorch_L2_MNIST.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()



model.fc1.weight.data.numpy()
plt.figure(figsize=(30, 50))
plt.imshow(model_mod5.fc2.weight.data.numpy(), cmap='gray')
plt.axis('off')  # Para no mostrar los ejes
plt.title('Pesos')
plt.show()

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(model_mod5.fc2.bias.data.numpy(), marker='o', linestyle='-')
plt.grid(True)
plt.show()

In [ ]:
import pickle

In [ ]:
import torch.nn.utils.prune as prune
import time
cont_i = 0
cont_j = 0
cont_k = 0
Perf_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151)), 10), dtype=float)
Time_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151)), 10), dtype=float)

for L in range(1,3):
  # for i_N in range(0, 30, 1):
  for i_N in range(0, 101, 1):
    print("Voy en: ", i_N)
    # for j_N in range(0, 50, 1):
    for j_N in range(0, 151, 1):
      for k_N in range(0,10):
        start_time = time.time()
        if i_N == 0 and j_N == 0:
          model.eval()
          correct = 0
          total = 0
          with torch.no_grad():
            L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
            _, predicted = torch.max(L_out.data, 1)
            total += all_labels.size(0)
            correct += (predicted == all_labels).sum().item()
          Perf_L1_L2_P[cont_i,cont_j,cont_k]= 100 * correct / total
        elif i_N == 0 and j_N != 0:
          model = torch.load(ruta_guardado_completo, weights_only=False)
          model_mod5 = copy.deepcopy(model)
          pruning_fraction_j = j_N/300
          # Aplicar pruning estructurado (por fila) a fc2
          prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
          with torch.no_grad():
            weight_mask = model_mod5.fc1.weight_mask
            if model_mod5.fc1.bias is not None:
                bias_mask = model_mod5.fc1.bias.data
                for i in range(weight_mask.size(0)):
                    if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                        bias_mask[i] = 0
                model_mod5.fc1.bias.data = bias_mask
          prune.remove(model_mod5.fc1, 'weight')
          model_mod5.eval()
          correct = 0
          total = 0
          with torch.no_grad():
            L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
            _, predicted = torch.max(L_out.data, 1)
            total += all_labels.size(0)
            correct += (predicted == all_labels).sum().item()
          Perf_L1_L2_P[cont_i,cont_j,cont_k]= 100 * correct / total
        else:
          model = torch.load(ruta_guardado_completo, weights_only=False)
          model_mod5 = copy.deepcopy(model)
          pruning_fraction_i = i_N/150
          prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=L, dim=0)
          with torch.no_grad():
            weight_mask = model_mod5.fc2.weight_mask
            if model_mod5.fc2.bias is not None:
              bias_mask = model_mod5.fc2.bias.data
              for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                  bias_mask[i] = 0
              model_mod5.fc2.bias.data = bias_mask
          prune.remove(model_mod5.fc2, 'weight')
          pruning_fraction_j = j_N/300
          # Aplicar pruning estructurado (por fila) a fc2
          prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
          with torch.no_grad():
            weight_mask = model_mod5.fc1.weight_mask
            if model_mod5.fc1.bias is not None:
                bias_mask = model_mod5.fc1.bias.data
                for i in range(weight_mask.size(0)):
                    if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                        bias_mask[i] = 0
                model_mod5.fc1.bias.data = bias_mask
          prune.remove(model_mod5.fc1, 'weight')
          model_mod5.eval()
          correct = 0
          total = 0
          with torch.no_grad():
            L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
            _, predicted = torch.max(L_out.data, 1)
            total += all_labels.size(0)
            correct += (predicted == all_labels).sum().item()
          Perf_L1_L2_P[cont_i,cont_j,cont_k]= 100 * correct / total
        Time_L1_L2_P[cont_i,cont_j,cont_k]= time.time() - start_time
        cont_k+=1
      cont_j+=1
      cont_k=0
    cont_i+=1
    cont_j=0
  cont_i=0
  cont_j=0
  cont_k=0

  rcParams.update({'font.size': 14})

  fig = plt.figure(figsize=(11, 12))
  ax = fig.add_subplot(111, projection='3d')

  x = np.linspace(0, Perf_L1_L2_P.shape[1] - 1, Perf_L1_L2_P.shape[1])
  y = np.linspace(0, Perf_L1_L2_P.shape[0] - 1, Perf_L1_L2_P.shape[0])
  X, Y = np.meshgrid(x, y)

  surf = ax.plot_surface(X, Y, Perf_L1_L2_P[:,:,5], cmap='viridis')

  # 4. Añadir etiquetas y título
  ax.set_xlabel('N 300', labelpad=20)
  ax.set_ylabel('N 150', labelpad=20)
  ax.set_zlabel('Performance', labelpad=10)

  ax.xaxis.set_major_locator(MultipleLocator(50))
  ax.yaxis.set_major_locator(MultipleLocator(25))

  # 5. Añadir una barra de color (opcional)
  #fig.colorbar(surf, shrink=0.5, aspect=10)

  # Establecer los ángulos de visión
  elev = 15   # Ángulo de elevación en grados (arriba/abajo)
  azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
  ax.view_init(elev=elev, azim=azim)

  nombre_archivo = DATA_ROOT + 'perf_prun_MNIST_pyTorch_L'+str(L)+'.png'
  plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight')

  # 6. Mostrar la gráfica
  plt.show()

  #root_N = DATA_ROOT + ''
  #nombre_archivo = "MNIST_PyTorch_L"+str(L)+".pkl"

  #with open(root_N+nombre_archivo, 'wb') as archivo:
  #    pickle.dump([oiplk<za, Time_L1_L2_P], archivo)

  #print(f"Variable guardada en {nombre_archivo}")

In [ ]:
model = torch.load(ruta_guardado_completo, weights_only=False)
model_mod5 = copy.deepcopy(model)

cont_i = 0
cont_j = 0
cont_k = 0
Perf_L1_L2_1 = np.zeros((len(range(0, -101, -1)), len(range(0, -151, -1)),10), dtype=float)
Time_L1_L2_NObSP = np.zeros((len(range(0, 101)), len(range(0, 151)), 10), dtype=float)

for i_N in range(0, -101, -1):
  print("Voy en: ", i_N)
  for j_N in range(0, -151, -1):
    for k_N in range(0,10):
      start_time = time.time()
      if i_N == 0 and j_N == 0:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
          L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
          _, predicted = torch.max(L_out.data, 1)
          total += all_labels.size(0)
          correct += (predicted == all_labels).sum().item()
        Perf_L1_L2_1[cont_i,cont_j,cont_k]= 100 * correct / total
      elif i_N == 0:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model_mod5 = copy.deepcopy(model)
        model_mod5, Perf_L1_L2_1[cont_i,cont_j,cont_k] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
      elif j_N == 0:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model_mod5 = copy.deepcopy(model)
        model_mod5, Perf_L1_L2_1[cont_i,cont_j,cont_k] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
      else:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model_mod5 = copy.deepcopy(model)
        model_mod5, _ = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
        model_mod5, Perf_L1_L2_1[cont_i,cont_j,cont_k] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
      Time_L1_L2_NObSP[cont_i,cont_j,cont_k]= time.time() - start_time
      cont_k+=1
    cont_j+=1
    cont_k=0
  cont_i+=1
  cont_j=0

rcParams.update({'font.size': 14})

fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2_1.shape[1] - 1, Perf_L1_L2_1.shape[1])
y = np.linspace(0, Perf_L1_L2_1.shape[0] - 1, Perf_L1_L2_1.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2_1[:,:,8], cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance', labelpad=10)

ax.xaxis.set_major_locator(MultipleLocator(50))
ax.yaxis.set_major_locator(MultipleLocator(25))

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

nombre_archivo = DATA_ROOT + 'perf_prun_MNIST_NObSP.png'
plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()

#print(f"Variable guardada en {nombre_archivo}")

#root_N = DATA_ROOT + ''
#nombre_archivo = "MNIST_NObSP.pkl"

#with open(root_N+nombre_archivo, 'wb') as archivo:
#    pickle.dump([Perf_L1_L2_1, Time_L1_L2_NObSP], archivo)

print(f"Variable guardada en {nombre_archivo}")

## 8. CostWall-clock time of the decomposition against that of magnitude pruning. NObSP is more expensive, and the question is whether the ranking it produces is worth the cost.

In [ ]:
Time_L1_L2_NObSP=Time_L1_L2_P

In [ ]:
import pickle

root_N = DATA_ROOT + ''
nombre_archivo = "MNIST_PyTorch_L1.pkl"

# Especifica el nombre del archivo pickle que quieres cargar
nombre_archivo = root_N+nombre_archivo  # Reemplaza con el nombre de tu archivo

try:
    with open(nombre_archivo, 'rb') as archivo:
        Perf_L1_L2_P1, Time_L1 = pickle.load(archivo)
    print(f"Variable cargada exitosamente desde {nombre_archivo}")
    # Ahora puedes trabajar con la variable_cargada
    # Por ejemplo, si guardaste una lista:
    # print(variable_cargada[0])
    # Si guardaste un diccionario:
    # print(variable_cargada['clave'])
except FileNotFoundError:
    print(f"Error: El archivo {nombre_archivo} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo pickle: {e}")

nombre_archivo = "MNIST_PyTorch_L2.pkl"

# Especifica el nombre del archivo pickle que quieres cargar
nombre_archivo = root_N+nombre_archivo  # Reemplaza con el nombre de tu archivo

try:
    with open(nombre_archivo, 'rb') as archivo:
        Perf_L1_L2_P1, Time_L2 = pickle.load(archivo)
    print(f"Variable cargada exitosamente desde {nombre_archivo}")
    # Ahora puedes trabajar con la variable_cargada
    # Por ejemplo, si guardaste una lista:
    # print(variable_cargada[0])
    # Si guardaste un diccionario:
    # print(variable_cargada['clave'])
except FileNotFoundError:
    print(f"Error: El archivo {nombre_archivo} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo pickle: {e}")

nombre_archivo = "MNIST_NObSP.pkl"

# Especifica el nombre del archivo pickle que quieres cargar
nombre_archivo = root_N+nombre_archivo  # Reemplaza con el nombre de tu archivo

try:
    with open(nombre_archivo, 'rb') as archivo:
        Perf_L1_L2_P1, Time_NOb = pickle.load(archivo)
    print(f"Variable cargada exitosamente desde {nombre_archivo}")
    # Ahora puedes trabajar con la variable_cargada
    # Por ejemplo, si guardaste una lista:
    # print(variable_cargada[0])
    # Si guardaste un diccionario:
    # print(variable_cargada['clave'])
except FileNotFoundError:
    print(f"Error: El archivo {nombre_archivo} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo pickle: {e}")

In [ ]:
v_T_L1 = Time_L1.reshape(-1)
v_T_L2 = Time_L1.reshape(-1)
v_T_NOb = Time_NOb.reshape(-1)

for i in range(25,100,25):
  print("L1: ",np.percentile(v_T_L1, i))
  print("L2: ",np.percentile(v_T_L2, i))
  print("NObSP: ",np.percentile(v_T_NOb, i))